In [5]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [6]:
!pip install nvcc4jupyter

In [7]:
%load_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [11]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <vector>
#include <iostream>
#include <chrono>
#include <ctime>
#include <cuda_runtime.h>


// ядро для вычисления суммы элементов массива
__global__ void sumKernel(int* gpuSum, int* gpuArray, unsigned int length)
{
  int globalId = blockIdx.x * blockDim.x + threadIdx.x; // глобальный индекс потока
  __shared__ int blockBuffer[256]; // общая память блока для редукции
  int currentValue = 0;

  // каждый поток берёт один элемент массива
  if (globalId < length)
  {
    currentValue = gpuArray[globalId];
  }
  // сохранение значения в shared memory
  blockBuffer[threadIdx.x] = currentValue;
  __syncthreads();

  // суммирование внутри блока
  for (int step = blockDim.x / 2; step > 0; step >>= 1)
  {
    if (threadIdx.x < step)
    {
      blockBuffer[threadIdx.x] += blockBuffer[threadIdx.x + step];
    }
    __syncthreads();
  }

  // первый поток блока добавляет частичную сумму в рез
  if (threadIdx.x == 0)
  {
    atomicAdd(gpuSum, blockBuffer[0]);
  }
}


// GPU
int get_sum_gpu(const std::vector<int>& numbers, double& gpuTime)
{
  auto gpuStart = std::chrono::high_resolution_clock::now();

  int* deviceArray;
  int* deviceSum;
  int hostSum = 0;
  int zeroValue = 0;

  // выделение памяти
  cudaMalloc((void**)&deviceArray, numbers.size() * sizeof(int));
  cudaMalloc((void**)&deviceSum, sizeof(int));
  // копирование данных на GPU
  cudaMemcpy(deviceSum, &zeroValue, sizeof(int), cudaMemcpyHostToDevice);
  cudaMemcpy(deviceArray, numbers.data(), numbers.size() * sizeof(int), cudaMemcpyHostToDevice);

  // параметры запуска ядра
  int threadsPerBlock = 256;
  int blocksPerGrid = (numbers.size() + threadsPerBlock - 1) / threadsPerBlock;
  // ядро
  sumKernel<<<blocksPerGrid, threadsPerBlock>>>(deviceSum, deviceArray, (unsigned int)numbers.size());
  // ожидание, синхронизация
  cudaDeviceSynchronize();

  // копирование реза на CPU
  cudaMemcpy(&hostSum, deviceSum, sizeof(int), cudaMemcpyDeviceToHost);

  auto gpuFinish = std::chrono::high_resolution_clock::now();
  gpuTime = std::chrono::duration<double>(gpuFinish - gpuStart).count();

  // освобождение памяти
  cudaFree(deviceArray);
  cudaFree(deviceSum);

  return hostSum;
}


// CPU
int get_sum_cpu(const std::vector<int>& numbers)
{
  int total = 0;
  // последовательное суммирование элементов
  for (int value : numbers)
  {
    total += value;
  }

  return total;
}


int main()
{
  int n = 1000000; // размер вектора и ниже его заполнение
  std::vector<int> data(n);

  srand(time(0));
  for (int i = 0; i < n; i++)
  {
    data[i] = rand() % 9 + 1;
  }

  auto cpuStart = std::chrono::high_resolution_clock::now();
  int cpuResult = get_sum_cpu(data);
  auto cpuFinish = std::chrono::high_resolution_clock::now();

  double cpuTime = std::chrono::duration<double>(cpuFinish - cpuStart).count();

  std::cout << "CPU /// sum: " << cpuResult << std::endl;
  std::cout << "CPU /// time: " << cpuTime << " sec" << std::endl;

  double gpuTime = 0.0;
  int gpuResult = get_sum_gpu(data, gpuTime);

  std::cout << "GPU /// sum: " << gpuResult << std::endl;
  std::cout << "GPU /// time: " << gpuTime << " sec" << std::endl;

  double speedup = cpuTime / gpuTime;
  std::cout << std::endl;
  std::cout << "speedup: " << speedup << std::endl;

  return 0;
}

CPU /// sum: 4996740
CPU /// time: 0.0113536 sec
GPU /// sum: 4996740
GPU /// time: 0.288752 sec

speedup: 0.0393195

